In [1]:
# 1. Uninstall any old versions
!pip uninstall -y anomaly_detector

# 2. Install the fresh version from GitHub
# We use --no-cache-dir to prevent Colab from using an old downloaded copy
!pip install --no-cache-dir git+https://github.com/ngriffen/AnomalyDetector.git
!pip install scikit-learn
# 3. IMPORTANT: Go to 'Runtime' -> 'Restart Session' after running this cell
# to make sure the new files are loaded into Python's memory!

Found existing installation: anomaly_detector 0.6.0
Uninstalling anomaly_detector-0.6.0:
  Successfully uninstalled anomaly_detector-0.6.0
  Cloning https://github.com/ngriffen/AnomalyDetector.git to /tmp/pip-req-build-naz707rq
  Running command git clone --filter=blob:none --quiet https://github.com/ngriffen/AnomalyDetector.git /tmp/pip-req-build-naz707rq
  Resolved https://github.com/ngriffen/AnomalyDetector.git to commit fdcbf9a732db5f91378d6c44421ff472e16e8e73
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for anomaly_detector: filename=anomaly_detector-0.6.0-py3-none-any.whl size=15783 sha256=9fc78a8cdf62b0b14128ebef2cbbf69c40e9861af9aaeb2805a90bd052e65b9d
  Stored in directory: /tmp/pip-ephem-wheel-cache-a54qmqfj/wheels/80/58/07/8ce4defe0bb677e25ce0b31fa99b9e29d7c7fcdfbed69bab6e
Successfully built anomaly_detector


In [2]:
import pandas as pd
import numpy as np
from anomaly_detector.detector import AnomalyDetector

# 1. Setup a more complex "Real-World" Stress Dataset
data = {
    # BASIC: 'F' and 'Unk' are rare
    "gender": (["Male"] * 50 + ["Female"] * 50 + ["F"] * 2 + ["Unk"] * 1),

    # BASIC: ~60% Nulls
    "middle_name": ([None] * 63 + ["Lee"] * 40),

    # BASIC: Statistical Outliers (IQR)
    "salary": [55000, 58000, 60000, 57000, 56000, 2000000, -500] + ([55000] * 96),

    # BASIC: Mixed types
    "tenure_years": [5, 10, "2 years", 8, 4] + ([5] * 98),

    # BASIC: Logical Violations (Rule: 0-100)
    "performance_score": [85, 92, 150, -5, 80] + ([85] * 98),

    # AUTO: These rows are designed to be "Multivariate Anomalies"
    # Row 7 and 8 have normal ages and normal salaries,
    # but the COMBINATION is statistically weird for the rest of the group.
    "age": [25, 26, 24, 25, 26, 25, 20, 70] + ([30] * 95),
}

df = pd.DataFrame(data)

# BASIC: Create 2 Duplicate Rows
df = pd.concat([df, df.iloc[[0, 1]]], ignore_index=True)

# 2. Configure Rules for Basic/Full modes
config = {
    "rare_threshold": 3,
    "null_threshold_pct": 15.0,
    "logical_rules": {
        "performance_score": {"min": 0, "max": 100},
        "salary": {"min": 0}
    },
    "patterns": {
        "gender": r"^(Male|Female)$" # Enforce full words only
    }
}

In [3]:
print("RUNNING STEP 1: BASIC MODE")
# Only runs the 6 Basic statistical checks
basic_report = AnomalyDetector.Basic(df, **config).run()
print(basic_report.summary())

RUNNING STEP 1: BASIC MODE
   ANOMALY REPORT [BASIC MODE]
  Dataset: 105 rows × 6 columns


1] Anomalies Detected (Rare Values)
--------------------------------------------------------------------------------
    - Col: 'age' | Value: 20         | Count: 1 (0.95%)
    - Col: 'age' | Value: 24         | Count: 1 (0.95%)
    - Col: 'age' | Value: 70         | Count: 1 (0.95%)
    - Col: 'age' | Value: 26         | Count: 3 (2.86%)
    - Col: 'gender' | Value: Unk        | Count: 1 (0.95%)
    - Col: 'gender' | Value: F          | Count: 2 (1.90%)
    - Col: 'performance_score' | Value: -5         | Count: 1 (0.95%)
    - Col: 'performance_score' | Value: 150        | Count: 1 (0.95%)
    - Col: 'performance_score' | Value: 80         | Count: 1 (0.95%)
    - Col: 'performance_score' | Value: 92         | Count: 2 (1.90%)
    - Col: 'salary' | Value: 57000      | Count: 1 (0.95%)
    - Col: 'salary' | Value: 60000      | Count: 1 (0.95%)
    - Col: 'salary' | Value: 2000000    | Count: 1 

In [4]:
print("\n\nRUNNING STEP 2: AUTO MODE")
# Uses ML (Isolation Forest) to find weird row combinations
# We set contamination higher to ensure it catches the age/salary combo anomalies
auto_report = AnomalyDetector.Auto(df, contamination=0.05).run()
print(auto_report.summary())



RUNNING STEP 2: AUTO MODE
   ANOMALY REPORT [AUTO MODE]
  Dataset: 105 rows × 6 columns


[AUTO] Unsupervised Multivariate Anomalies
--------------------------------------------------------------------------------
  ! Detected 6 rows with anomalous combinations.
    - Row    2: [Suspect: performance_score]
               Data: gender: Male, middle_name: None, salary: 60000, tenure_years: 2 years, performance_score: 150, age: 24
    - Row    3: [Suspect: performance_score]
               Data: gender: Male, middle_name: None, salary: 57000, tenure_years: 8, performance_score: -5, age: 25
    - Row    4: [Complex Interaction: age*, performance_score*]
               Data: gender: Male, middle_name: None, salary: 56000, tenure_years: 4, performance_score: 80, age: 26
    - Row    5: [Suspect: salary]
               Data: gender: Male, middle_name: None, salary: 2000000, tenure_years: 5, performance_score: 85, age: 25
    - Row    6: [Suspect: age]
               Data: gender: Male, midd

In [5]:
print("\n\nRUNNING STEP 3: FULL MODE")
# Runs EVERYTHING: Rules + Statistics + Machine Learning
full_report = AnomalyDetector.Full(df, **config).run()
print(full_report.summary())



RUNNING STEP 3: FULL MODE
   ANOMALY REPORT [FULL MODE]
  Dataset: 105 rows × 6 columns


[AUTO] Unsupervised Multivariate Anomalies
--------------------------------------------------------------------------------
  ! Detected 3 rows with anomalous combinations.
    - Row    3: [Suspect: performance_score]
               Data: gender: Male, middle_name: None, salary: 57000, tenure_years: 8, performance_score: -5, age: 25
    - Row    5: [Suspect: salary]
               Data: gender: Male, middle_name: None, salary: 2000000, tenure_years: 5, performance_score: 85, age: 25
    - Row    7: [Suspect: age]
               Data: gender: Male, middle_name: None, salary: 55000, tenure_years: 5, performance_score: 85, age: 70

1] Anomalies Detected (Rare Values)
--------------------------------------------------------------------------------
    - Col: 'age' | Value: 20         | Count: 1 (0.95%)
    - Col: 'age' | Value: 24         | Count: 1 (0.95%)
    - Col: 'age' | Value: 70         | Cou